# 01｜第 2 節：讓 AI 猜會不會及格

**讀資料 → 分組 → 分類 → 看 Accuracy；選做 K-means**。第 1 格已寫好匯入模組與讀取資料，直接執行；第 2、3 格依 Markdown 範例輸入到空白儲存格；最後兩格 Optional 可直接執行。CSV 請與 Notebook 放在同一資料夾。

### 資料集欄位（500 筆）

| 欄位 | 意義 | 本次用途 |
|---|---|---|
| `study_hours` | 每天讀書幾小時，有空白與一筆 60 小時 | 第 1 節的輸入 X |
| `quiz_score` | 小考分數（0～100） | 第 1 節答案 y |
| `practice_questions` | 每天練習幾題，有空白 | 第 2 節的輸入 X |
| `passed` | 是否及格：0＝未及格、1＝及格 | 第 2 節答案 y |
| 其餘欄位 | 學號、學習方式、出席率、睡眠、努力程度 | 本次不用 |

`X` 是模型看到的資料，`y` 是要猜的答案。不要把 `quiz_score` 或 `passed` 偷放到同一個任務的 `X`。

### 步驟 1/3｜匯入模組與讀取資料（直接執行）

這一節只用 `practice_questions`（練習題數）預測 `passed`（0／1）。

**這一格已經寫好，直接按 Shift + Enter 執行，不需要重新輸入。**

In [ ]:
# Import pandas for loading tabular data.
# 匯入 pandas 讀取表格資料。
import pandas as pd

# Import matplotlib for optional graphs.
# 匯入畫圖套件，供選做實驗使用。
import matplotlib.pyplot as plt

# Import the training/test split function.
# 匯入訓練與測試資料切分工具。
from sklearn.model_selection import train_test_split

# Import a simple pass/fail classification model.
# 匯入判斷及格與否的分類模型。
from sklearn.linear_model import LogisticRegression

# Import accuracy to count correct predictions.
# 匯入 Accuracy 正確率評估工具。
from sklearn.metrics import accuracy_score

# Import the optional clustering algorithm.
# 匯入選做分群實驗的 K-means。
from sklearn.cluster import KMeans

# Load the same provided CSV dataset.
# 讀取與上一節相同的 CSV 資料。
df = pd.read_csv('01_機器學習_學生練習資料_500筆.csv')

# Display the practice count and answer.
# 預覽練習題數與是否及格。
print(df[['practice_questions', 'passed']].head())

### 必做 2/3｜把練習資料分兩組

先分訓練與測試；空白的練習題數用訓練組中位數補齊。

```python
# Use the number of solved questions as input.
# 把練習題數當作模型輸入。
X = df[['practice_questions']]

# Use pass/fail as the label.
# 把及格 1／未及格 0 當成答案。
y = df['passed']

# Separate training and testing examples.
# 保留 20% 測試資料，兩類比例盡量一致。
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Calculate a training-only median for gaps.
# 只使用訓練組決定空白值的中位數。
median_questions = X_train['practice_questions'].median()

# Fill the missing training question counts.
# 補齊訓練組空白的練習題數。
X_train = X_train.fillna(median_questions)

# Fill test gaps using the same median.
# 測試組也使用相同的中位數。
X_test = X_test.fillna(median_questions)
```

In [ ]:
# 將上方「必做 2/3」範例輸入此格，再按 Shift + Enter。

### 必做 3/3｜訓練分類並算 Accuracy

Accuracy 就是「猜對幾筆 ÷ 全部幾筆」；先不學較複雜的分類評估公式。

```python
# Create a simple binary classifier.
# 建立簡單的二元分類模型。
model = LogisticRegression()

# Learn from the training split only.
# 只使用訓練組學習。
model.fit(X_train, y_train)

# Predict pass/fail on the hidden test set.
# 對測試資料預測及格 1 或未及格 0。
pred = model.predict(X_test)

# Measure the proportion of correct answers.
# 計算答對的比例 Accuracy。
accuracy = accuracy_score(y_test, pred)

# Display correct answers as a percentage.
# 將正確率以百分比顯示。
print(f'Accuracy = {accuracy:.0%}')
```

In [ ]:
# 將上方「必做 3/3」範例輸入此格，再按 Shift + Enter。

In [ ]:
# Optional experiment: 畫出及格機率曲線
# 選做實驗：畫出及格機率曲線
# Create example counts from zero to thirty.
# 建立 0～30 題的練習題數。
new_data = pd.DataFrame({'practice_questions': range(31)})

# Predict pass probabilities for those counts.
# 取得不同題數對應的預測及格機率。
probabilities = model.predict_proba(new_data)[:, 1]

# Draw the predicted probability curve.
# 畫出題數越多，機率如何變化的曲線。
plt.plot(new_data['practice_questions'], probabilities, linewidth=3)

# Display the simple one-half decision threshold.
# 畫出 50% 的分類參考線。
plt.axhline(0.5, color='orange', linestyle='--')

# Label the horizontal axis.
# 標示橫軸為練習題數。
plt.xlabel('Practice questions')

# Label the vertical axis.
# 標示縱軸為及格機率。
plt.ylabel('Pass probability')

# Display the probability chart.
# 顯示及格機率曲線。
plt.show()


In [ ]:
# Optional experiment: K-means 將學習紀錄分成兩群
# 選做實驗：K-means 將學習紀錄分成兩群
# Select the two numerical columns for grouping.
# 使用讀書時數與練習題數來做分群。
points = df[['study_hours', 'practice_questions']].copy()

# Exclude the impossible daily 60-hour value.
# 將一天 60 小時的錯誤紀錄標為缺漏。
points.loc[points['study_hours'] > 24, 'study_hours'] = float('nan')

# Fill missing values for this optional exploration.
# 為選做實驗補齊兩欄的空白資料。
points = points.fillna(points.median())

# Group examples into two clusters without labels.
# 不使用及格標籤，把資料分成兩群。
groups = KMeans(n_clusters=2, random_state=42, n_init=10).fit(points)

# Plot original units rather than scaled values.
# 直接使用原始單位繪製分群散佈圖。
plt.scatter(points['study_hours'], points['practice_questions'], c=groups.labels_, s=12, alpha=0.5)

# Show the two cluster centers with crosses.
# 用 X 記號標出兩個群中心。
plt.scatter(groups.cluster_centers_[:, 0], groups.cluster_centers_[:, 1], marker='x', s=180, color='black')

# Label the first feature axis.
# 標示讀書時數的橫軸。
plt.xlabel('Study hours')

# Label the second feature axis.
# 標示練習題數的縱軸。
plt.ylabel('Practice questions')

# Display the clustering graph.
# 顯示分群散佈圖。
plt.show()
